# Self-Contained Colab Training Notebook

This notebook is designed for Google Colab and is fully self-contained.

Referenced Kaggle notebook:
- `krooz0/deep-fake-detection-on-images-and-videos`

What was verified from that notebook:
- `cellId=3` is only the problem-statement markdown.
- image training uses:
  - a custom CNN baseline
  - then Xception transfer learning

This Colab notebook follows that same direction, but in a cleaner reusable form.

Supported dataset layouts:
- `data/real`, `data/fake`
- `data/train/real`, `data/train/fake`, `data/val/real`, `data/val/fake`

Optional features:
- mount Google Drive
- authenticate Kaggle
- pull the Kaggle kernel for reference
- optionally download a Kaggle dataset
- train your own model with `xception`, `vgg16`, `mobilenetv2`, `efficientnetb0`, or `custom_cnn`


In [ ]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'kaggle',
        'tensorflow>=2.16,<2.21',
        'matplotlib',
        'pandas',
        'scikit-learn'
    ],
    check=True,
)
print('Dependencies installed.')


In [ ]:
from pathlib import Path

MOUNT_DRIVE = False
KAGGLE_JSON = Path('/content/kaggle.json')
KAGGLE_DATASET = None  # e.g. 'your-owner/your-dataset'
KAGGLE_KERNEL = 'krooz0/deep-fake-detection-on-images-and-videos'

DATA_DIR = Path('/content/data')
DATASET_ARCHIVE = None  # e.g. Path('/content/my_dataset.zip')
DATASET_SOURCE_DIR = None  # e.g. Path('/content/drive/MyDrive/deepfake-data')
WORK_DIR = Path('/content/work')
OUTPUT_DIR = Path('/content/output')

BACKBONE = 'xception'  # xception, vgg16, mobilenetv2, efficientnetb0, custom_cnn
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 8
FINETUNE_EPOCHS = 4
LEARNING_RATE = 1e-4
SEED = 42

WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR   =', DATA_DIR)
print('DATASET_ARCHIVE =', DATASET_ARCHIVE)
print('DATASET_SOURCE_DIR =', DATASET_SOURCE_DIR)
print('WORK_DIR   =', WORK_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('BACKBONE   =', BACKBONE)


In [ ]:
import json
import os
import shutil
import subprocess
import textwrap
from pathlib import Path
from zipfile import ZipFile


def run(cmd, cwd=None):
    print('\n$', ' '.join(map(str, cmd)))
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)


def maybe_mount_drive():
    if not MOUNT_DRIVE:
        return
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive')
    except Exception as exc:
        raise RuntimeError(f'Could not mount Drive: {exc}')


def configure_kaggle():
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    target = kaggle_dir / 'kaggle.json'

    if os.getenv('KAGGLE_USERNAME') and os.getenv('KAGGLE_KEY'):
        payload = {
            'username': os.environ['KAGGLE_USERNAME'],
            'key': os.environ['KAGGLE_KEY'],
        }
        target.write_text(json.dumps(payload))
        target.chmod(0o600)
        print('Configured Kaggle from environment variables.')
        return

    if KAGGLE_JSON.exists():
        shutil.copy2(KAGGLE_JSON, target)
        target.chmod(0o600)
        print('Configured Kaggle from kaggle.json file.')
        return

    raise FileNotFoundError(
        'Kaggle credentials not found. Upload kaggle.json to /content/kaggle.json '
        'or set KAGGLE_USERNAME and KAGGLE_KEY.'
    )


def pull_kernel(kernel_slug, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    run(['kaggle', 'kernels', 'pull', kernel_slug, '-m', '-p', str(output_dir)])
    assets = sorted(output_dir.iterdir())
    print('Kernel assets:')
    for asset in assets:
        print('-', asset)
    return assets


def maybe_download_dataset(dataset_slug, data_dir):
    if not dataset_slug:
        print('No Kaggle dataset slug provided. Skipping dataset download.')
        return
    data_dir.mkdir(parents=True, exist_ok=True)
    run(['kaggle', 'datasets', 'download', '-d', dataset_slug, '-p', str(data_dir)])
    zip_files = sorted(data_dir.glob('*.zip'))
    if not zip_files:
        raise FileNotFoundError('No downloaded dataset zip file found.')
    archive = zip_files[-1]
    print('Extracting', archive)
    with ZipFile(archive, 'r') as zf:
        zf.extractall(data_dir)


def maybe_extract_dataset_archive(dataset_archive, data_dir):
    if not dataset_archive:
        return
    archive = Path(dataset_archive)
    if not archive.exists():
        raise FileNotFoundError(f'Dataset archive not found: {archive}')
    data_dir.mkdir(parents=True, exist_ok=True)
    print('Extracting dataset archive', archive)
    with ZipFile(archive, 'r') as zf:
        zf.extractall(data_dir)


def maybe_copy_dataset_dir(dataset_source_dir, data_dir):
    if not dataset_source_dir:
        return
    source_dir = Path(dataset_source_dir)
    if not source_dir.exists():
        raise FileNotFoundError(f'Dataset source directory not found: {source_dir}')
    if source_dir.resolve() == data_dir.resolve():
        print('DATASET_SOURCE_DIR already matches DATA_DIR. No copy needed.')
        return
    data_dir.mkdir(parents=True, exist_ok=True)
    print('Copying dataset contents from', source_dir, 'to', data_dir)
    for item in source_dir.iterdir():
        target = data_dir / item.name
        if target.exists():
            continue
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)


def detect_dataset_layout(data_dir, raise_on_missing=False):
    train_dir = data_dir / 'train'
    val_dir = data_dir / 'val'

    if (train_dir / 'real').exists() and (train_dir / 'fake').exists():
        layout = {'train': train_dir}
        if (val_dir / 'real').exists() and (val_dir / 'fake').exists():
            layout['val'] = val_dir
        return layout

    if (data_dir / 'real').exists() and (data_dir / 'fake').exists():
        return {'all': data_dir}

    for path in sorted(data_dir.rglob('*')):
        if not path.is_dir():
            continue
        names = {child.name.lower() for child in path.iterdir() if child.is_dir()}
        if {'real', 'fake'}.issubset(names):
            return {'all': path}

    if raise_on_missing:
        raise FileNotFoundError(
            f'Could not detect dataset layout under {data_dir}. '
            'Expected `real/` + `fake/` or `train/real`, `train/fake`.'
        )
    return None


def print_dataset_help(data_dir):
    print(textwrap.dedent(
        f'''
        Dataset not found under {data_dir}.

        Configure one of these in the config cell, then rerun the setup cell:
        1. KAGGLE_DATASET = 'owner/dataset-slug'
        2. DATASET_ARCHIVE = Path('/content/your_dataset.zip')
        3. DATASET_SOURCE_DIR = Path('/content/drive/MyDrive/your_dataset_folder')

        Expected folder layout:
        - /content/data/real
        - /content/data/fake
          or
        - /content/data/train/real
        - /content/data/train/fake
        - /content/data/val/real
        - /content/data/val/fake
        '''
    ).strip())


def maybe_prepare_dataset(data_dir):
    existing_layout = detect_dataset_layout(data_dir)
    if existing_layout is not None:
        print('Detected existing dataset layout:', existing_layout)
        return existing_layout

    maybe_copy_dataset_dir(DATASET_SOURCE_DIR, data_dir)
    layout = detect_dataset_layout(data_dir)
    if layout is not None:
        print('Detected dataset after copying source directory:', layout)
        return layout

    maybe_extract_dataset_archive(DATASET_ARCHIVE, data_dir)
    layout = detect_dataset_layout(data_dir)
    if layout is not None:
        print('Detected dataset after extracting archive:', layout)
        return layout

    maybe_download_dataset(KAGGLE_DATASET, data_dir)
    layout = detect_dataset_layout(data_dir)
    if layout is not None:
        print('Detected dataset after Kaggle download:', layout)
        return layout

    print_dataset_help(data_dir)
    return None


maybe_mount_drive()
configure_kaggle()
kernel_assets = pull_kernel(KAGGLE_KERNEL, WORK_DIR / 'kaggle_kernel')
dataset_layout = maybe_prepare_dataset(DATA_DIR)
DATASET_READY = dataset_layout is not None
print('DATASET_READY =', DATASET_READY)


In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

tf.random.set_seed(SEED)
np.random.seed(SEED)


def make_datasets(layout, image_size, batch_size, seed):
    dims = (image_size, image_size)

    if 'train' in layout:
        train_ds = tf.keras.utils.image_dataset_from_directory(
            layout['train'],
            labels='inferred',
            label_mode='binary',
            image_size=dims,
            batch_size=batch_size,
            shuffle=True,
            seed=seed,
        )
        if 'val' not in layout:
            raise ValueError('Validation directory missing for explicit train/val layout.')
        val_ds = tf.keras.utils.image_dataset_from_directory(
            layout['val'],
            labels='inferred',
            label_mode='binary',
            image_size=dims,
            batch_size=batch_size,
            shuffle=False,
        )
    else:
        train_ds = tf.keras.utils.image_dataset_from_directory(
            layout['all'],
            labels='inferred',
            label_mode='binary',
            image_size=dims,
            batch_size=batch_size,
            validation_split=0.2,
            subset='training',
            seed=seed,
        )
        val_ds = tf.keras.utils.image_dataset_from_directory(
            layout['all'],
            labels='inferred',
            label_mode='binary',
            image_size=dims,
            batch_size=batch_size,
            validation_split=0.2,
            subset='validation',
            seed=seed,
        )

    class_names = list(train_ds.class_names)
    autotune = tf.data.AUTOTUNE
    train_ds = train_ds.prefetch(autotune)
    val_ds = val_ds.prefetch(autotune)
    return train_ds, val_ds, class_names


train_ds = None
val_ds = None
class_names = []

if not DATASET_READY:
    print('Dataset setup incomplete. Configure the dataset in the config cell, rerun setup, then rerun this cell.')
else:
    train_ds, val_ds, class_names = make_datasets(
        layout=dataset_layout,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        seed=SEED,
    )

    (OUTPUT_DIR / 'class_names.json').write_text(json.dumps(class_names, indent=2))
    print('Class names:', class_names)
    print('Train batches:', tf.data.experimental.cardinality(train_ds).numpy())
    print('Val batches:', tf.data.experimental.cardinality(val_ds).numpy())


In [ ]:
if not DATASET_READY:
    print('Preview skipped because the dataset is not configured yet.')
else:
    plt.figure(figsize=(12, 12))
    for images, labels in train_ds.take(1):
        for i in range(min(9, len(images))):
            plt.subplot(3, 3, i + 1)
            plt.imshow(images[i].numpy().astype('uint8'))
            label_idx = int(labels[i].numpy()[0])
            plt.title(class_names[label_idx])
            plt.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
def build_model(backbone, image_size, learning_rate):
    backbone = backbone.lower()
    input_shape = (image_size, image_size, 3)

    if backbone == 'custom_cnn':
        inputs = tf.keras.Input(shape=input_shape)
        x = tf.keras.layers.Rescaling(1.0 / 255.0)(inputs)
        x = tf.keras.layers.Conv2D(64, 7, activation='relu', padding='same', kernel_initializer='he_normal')(x)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same', kernel_initializer='he_normal')(x)
        x = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same', kernel_initializer='he_normal')(x)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Flatten()(x)
        x = tf.keras.layers.Dense(128, activation='relu', kernel_initializer='he_normal')(x)
        x = tf.keras.layers.Dropout(0.5)(x)
        x = tf.keras.layers.Dense(64, activation='relu', kernel_initializer='he_normal')(x)
        x = tf.keras.layers.Dropout(0.5)(x)
        outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='fake_probability')(x)
        model = tf.keras.Model(inputs, outputs, name='custom_cnn_deepfake')
        base_model = None
    else:
        if backbone == 'xception':
            base_model = tf.keras.applications.Xception(include_top=False, weights='imagenet', input_shape=input_shape)
            preprocess = tf.keras.applications.xception.preprocess_input
        elif backbone == 'vgg16':
            base_model = tf.keras.applications.VGG16(include_top=False, weights='imagenet', input_shape=input_shape)
            preprocess = tf.keras.applications.vgg16.preprocess_input
        elif backbone == 'mobilenetv2':
            base_model = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=input_shape)
            preprocess = tf.keras.applications.mobilenet_v2.preprocess_input
        elif backbone == 'efficientnetb0':
            base_model = tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=input_shape)
            preprocess = tf.keras.applications.efficientnet.preprocess_input
        else:
            raise ValueError(f'Unsupported backbone: {backbone}')

        base_model.trainable = False
        inputs = tf.keras.Input(shape=input_shape)
        x = tf.keras.layers.Lambda(preprocess, name='preprocess')(inputs)
        x = base_model(x, training=False)
        x = tf.keras.layers.GlobalAveragePooling2D()(x)
        x = tf.keras.layers.Dropout(0.35)(x)
        x = tf.keras.layers.Dense(256, activation='relu')(x)
        x = tf.keras.layers.Dropout(0.25)(x)
        outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='fake_probability')(x)
        model = tf.keras.Model(inputs, outputs, name=f'{backbone}_deepfake_binary')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    return model, base_model


def merge_histories(*histories):
    merged = {}
    for history in histories:
        if history is None:
            continue
        for key, values in history.history.items():
            merged.setdefault(key, []).extend(list(values))
    return merged


model, base_model = build_model(BACKBONE, IMAGE_SIZE, LEARNING_RATE)
model.summary()


In [ ]:
if not DATASET_READY:
    print('Training skipped because the dataset is not configured yet.')
else:
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(OUTPUT_DIR / 'best_model.keras'),
            monitor='val_auc',
            mode='max',
            save_best_only=True,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc',
            mode='max',
            patience=3,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=2,
            min_lr=1e-6,
        ),
    ]

    print('Starting warmup training...')
    warmup_history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
    )

    finetune_history = None
    if base_model is not None and FINETUNE_EPOCHS > 0:
        print('Starting fine-tuning...')
        base_model.trainable = True
        if len(base_model.layers) > 20:
            for layer in base_model.layers[:-20]:
                layer.trainable = False

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE / 10.0),
            loss='binary_crossentropy',
            metrics=[
                'accuracy',
                tf.keras.metrics.AUC(name='auc'),
                tf.keras.metrics.Precision(name='precision'),
                tf.keras.metrics.Recall(name='recall'),
            ],
        )
        finetune_history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS + FINETUNE_EPOCHS,
            initial_epoch=len(warmup_history.history.get('loss', [])),
            callbacks=callbacks,
        )

    history = merge_histories(warmup_history, finetune_history)
    eval_metrics = model.evaluate(val_ds, return_dict=True)
    model.save(OUTPUT_DIR / 'final_model.keras')
    (OUTPUT_DIR / 'history.json').write_text(json.dumps(history, indent=2))
    (OUTPUT_DIR / 'metrics.json').write_text(json.dumps(eval_metrics, indent=2))

    summary = {
        'kaggle_kernel': KAGGLE_KERNEL,
        'kernel_assets': [str(path) for path in kernel_assets],
        'dataset_layout': {key: str(value) for key, value in dataset_layout.items()},
        'class_names': class_names,
        'backbone': BACKBONE,
        'image_size': IMAGE_SIZE,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'finetune_epochs': FINETUNE_EPOCHS,
        'eval_metrics': eval_metrics,
        'best_model': str(OUTPUT_DIR / 'best_model.keras'),
        'final_model': str(OUTPUT_DIR / 'final_model.keras'),
    }
    (OUTPUT_DIR / 'run_summary.json').write_text(json.dumps(summary, indent=2))

    print('Training complete.')
    print(json.dumps(summary, indent=2))


In [ ]:
import json
import matplotlib.pyplot as plt

history_path = OUTPUT_DIR / 'history.json'
metrics_path = OUTPUT_DIR / 'metrics.json'

if not history_path.exists() or not metrics_path.exists():
    print('No training artifacts found yet. Configure a dataset and run the training cell first.')
else:
    history = json.loads(history_path.read_text())
    metrics = json.loads(metrics_path.read_text())

    epochs_range = range(1, len(history.get('loss', [])) + 1)

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history.get('accuracy', []), label='train_accuracy')
    plt.plot(epochs_range, history.get('val_accuracy', []), label='val_accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, history.get('loss', []), label='train_loss')
    plt.plot(epochs_range, history.get('val_loss', []), label='val_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

    print('Validation metrics:')
    print(json.dumps(metrics, indent=2))
